# Advanced Problems: Parameter Defaults — Mutable Defaults and Memoization

This notebook contains advanced practice problems with full solutions on Python mutable default arguments, shared state, aliasing, defensive copying, and memoization.

In [1]:
from functools import lru_cache
from copy import deepcopy
import inspect
import math

## Core Principle

Function defaults are evaluated once, when the function is defined. If a default value is mutable, the same object is reused across calls.

## Problem 1 — Grocery List Bug

The following function is intended to start a new grocery list whenever `grocery_list` is omitted. Explain the bug and fix it.

In [2]:
def add_item_bad(name, quantity, unit, grocery_list=[]):
    item = f'{name} ({quantity} {unit})'
    grocery_list.append(item)
    return grocery_list

store_1 = add_item_bad('bananas', 2, 'units')
add_item_bad('grapes', 1, 'bunch', store_1)

store_2 = add_item_bad('milk', 1, 'gallon')

print('store_1:', store_1)
print('store_2:', store_2)
print('same object?', store_1 is store_2)

store_1: ['bananas (2 units)', 'grapes (1 bunch)', 'milk (1 gallon)']
store_2: ['bananas (2 units)', 'grapes (1 bunch)', 'milk (1 gallon)']
same object? True


### Solution 1

`[]` is created once when `add_item_bad` is defined. Every call that omits `grocery_list` uses the same list.

The fix is to use `None` as the default and create a fresh list inside the function.

In [3]:
def add_item(name, quantity, unit, grocery_list=None):
    if grocery_list is None:
        grocery_list = []
    item = f'{name} ({quantity} {unit})'
    grocery_list.append(item)
    return grocery_list

store_1 = add_item('bananas', 2, 'units')
add_item('grapes', 1, 'bunch', store_1)

store_2 = add_item('milk', 1, 'gallon')

print('store_1:', store_1)
print('store_2:', store_2)
print('same object?', store_1 is store_2)

store_1: ['bananas (2 units)', 'grapes (1 bunch)']
store_2: ['milk (1 gallon)']
same object? False


## Problem 2 — The `if not grocery_list` Trap

This version looks reasonable, but it contains a subtle bug.

Explain the bug.

In [4]:
def add_item_buggy(name, quantity, unit, grocery_list=None):
    if not grocery_list:
        grocery_list = []
    item = f'{name} ({quantity} {unit})'
    grocery_list.append(item)
    return grocery_list

my_list = []
result = add_item_buggy('eggs', 12, 'units', my_list)

print('my_list:', my_list)
print('result:', result)
print('same object?', my_list is result)

my_list: []
result: ['eggs (12 units)']
same object? False


### Solution 2

`if not grocery_list` treats an explicitly supplied empty list as missing. That breaks aliasing expectations: the caller supplied `my_list`, but the function replaced it with a new list.

Use `is None` when `None` means omitted.

In [5]:
def add_item_safe(name, quantity, unit, grocery_list=None):
    if grocery_list is None:
        grocery_list = []
    item = f'{name} ({quantity} {unit})'
    grocery_list.append(item)
    return grocery_list

my_list = []
result = add_item_safe('eggs', 12, 'units', my_list)

print('my_list:', my_list)
print('result:', result)
print('same object?', my_list is result)

my_list: ['eggs (12 units)']
result: ['eggs (12 units)']
same object? True


## Problem 3 — Inspect the Hidden Shared Object

Use introspection to prove that the bad function stores the default list on the function object itself.

In [6]:
def demo(a=[]):
    a.append('x')
    return a

print('defaults before:', demo.__defaults__)
demo()
demo()
print('defaults after:', demo.__defaults__)
print('signature:', inspect.signature(demo))

defaults before: ([],)
defaults after: (['x', 'x'],)
signature: (a=['x', 'x'])


### Solution 3

`demo.__defaults__` contains the actual default objects used for positional parameters. Because the default list is mutable, appending through the function changes the object stored in `__defaults__`.

## Problem 4 — Defensive Copying

Write `add_item_copying` so that:

- omitted `grocery_list` starts a new list
- supplied `grocery_list` is not mutated
- the function returns the updated copy

### Solution 4

In [7]:
def add_item_copying(name, quantity, unit, grocery_list=None):
    if grocery_list is None:
        grocery_list = []
    else:
        grocery_list = list(grocery_list)

    item = f'{name} ({quantity} {unit})'
    grocery_list.append(item)
    return grocery_list

original = ['bananas (2 units)']
new_list = add_item_copying('milk', 1, 'gallon', original)

print('original:', original)
print('new_list:', new_list)
print('same object?', original is new_list)

original: ['bananas (2 units)']
new_list: ['bananas (2 units)', 'milk (1 gallon)']
same object? False


## Problem 5 — Nested Mutable Defaults

Fix this function. The default value contains nested mutable data.

In [8]:
def create_cart_bad(items=[], metadata={'source': 'manual', 'discounts': []}):
    return {'items': items, 'metadata': metadata}

cart_1 = create_cart_bad()
cart_2 = create_cart_bad()

cart_1['items'].append('apples')
cart_1['metadata']['discounts'].append('WELCOME10')

print('cart_1:', cart_1)
print('cart_2:', cart_2)

cart_1: {'items': ['apples'], 'metadata': {'source': 'manual', 'discounts': ['WELCOME10']}}
cart_2: {'items': ['apples'], 'metadata': {'source': 'manual', 'discounts': ['WELCOME10']}}


### Solution 5

Use `None` defaults. For nested user-provided structures, use `deepcopy` if the returned object should be isolated from external mutation.

In [9]:
def create_cart(items=None, metadata=None):
    if items is None:
        items = []
    else:
        items = list(items)

    if metadata is None:
        metadata = {'source': 'manual', 'discounts': []}
    else:
        metadata = deepcopy(metadata)

    return {'items': items, 'metadata': metadata}

cart_1 = create_cart()
cart_2 = create_cart()

cart_1['items'].append('apples')
cart_1['metadata']['discounts'].append('WELCOME10')

print('cart_1:', cart_1)
print('cart_2:', cart_2)

cart_1: {'items': ['apples'], 'metadata': {'source': 'manual', 'discounts': ['WELCOME10']}}
cart_2: {'items': [], 'metadata': {'source': 'manual', 'discounts': []}}


## Problem 6 — Explicit `None` Has Meaning

Design `prepare_order(items=...)` with these rules:

- omitted `items` means start with an empty list
- `items=None` means the order is intentionally itemless
- supplied iterable means copy the iterable into a list

### Solution 6

When `None` is meaningful, use a private sentinel object to detect omission.

In [10]:
_MISSING = object()

def prepare_order(items=_MISSING):
    if items is _MISSING:
        items = []
    elif items is None:
        items = None
    else:
        items = list(items)

    return {'items': items}

print(prepare_order())
print(prepare_order(None))
print(prepare_order(['milk', 'eggs']))

{'items': []}
{'items': None}
{'items': ['milk', 'eggs']}


## Problem 7 — Intentional Mutable Default for Memoization

The following factorial function recalculates values repeatedly. Add memoization using an intentional mutable default.

In [11]:
def factorial_slow(n):
    if n < 0:
        raise ValueError('n must be non-negative')
    if n < 2:
        return 1
    print(f'calculating {n}!')
    return n * factorial_slow(n - 1)

print(factorial_slow(5))
print(factorial_slow(5))

calculating 5!
calculating 4!
calculating 3!
calculating 2!
120
calculating 5!
calculating 4!
calculating 3!
calculating 2!
120


### Solution 7

Here the mutable default is intentional. The cache persists between calls. This can be useful, but it creates hidden shared state.

In [12]:
def factorial_memo(n, cache={0: 1, 1: 1}):
    if n < 0:
        raise ValueError('n must be non-negative')
    if n not in cache:
        print(f'calculating {n}!')
        cache[n] = n * factorial_memo(n - 1)
    return cache[n]

print(factorial_memo(5))
print(factorial_memo(5))
print(factorial_memo(7))
print('cache:', factorial_memo.__defaults__[0])

calculating 5!
calculating 4!
calculating 3!
calculating 2!
120
120
calculating 7!
calculating 6!
5040
cache: {0: 1, 1: 1, 2: 2, 3: 6, 4: 24, 5: 120, 6: 720, 7: 5040}


## Problem 8 — Safer Memoization with `lru_cache`

Rewrite factorial memoization using `functools.lru_cache`.

### Solution 8

In [13]:
@lru_cache(maxsize=None)
def factorial_cached(n):
    if n < 0:
        raise ValueError('n must be non-negative')
    if n < 2:
        return 1
    print(f'calculating {n}!')
    return n * factorial_cached(n - 1)

print(factorial_cached(5))
print(factorial_cached(5))
print(factorial_cached(7))
print(factorial_cached.cache_info())

calculating 5!
calculating 4!
calculating 3!
calculating 2!
120
120
calculating 7!
calculating 6!
5040
CacheInfo(hits=2, misses=7, maxsize=None, currsize=7)


## Problem 9 — Cache Pollution Bug

The following memoized function accepts an optional cache. Explain why this design is risky.

In [14]:
def power_bad(base, exponent, cache={}):
    key = exponent
    if key not in cache:
        cache[key] = base ** exponent
    return cache[key]

print(power_bad(2, 3))
print(power_bad(10, 3))

8
8


### Solution 9

The cache key only uses `exponent`, but the result also depends on `base`. The second call incorrectly reuses the result of `2 ** 3` for `10 ** 3`.

The cache key must include all inputs that affect the result.

In [15]:
def power_good(base, exponent, cache={}):
    key = (base, exponent)
    if key not in cache:
        cache[key] = base ** exponent
    return cache[key]

print(power_good(2, 3))
print(power_good(10, 3))
print(power_good.__defaults__[0])

8
1000
{(2, 3): 8, (10, 3): 1000}


## Problem 10 — Testing for Shared State

Write a small test that catches accidental shared mutable defaults in a function like `add_item`.

### Solution 10

In [16]:
def assert_independent_lists(func):
    a = func('apples', 2, 'units')
    b = func('milk', 1, 'gallon')

    assert a is not b, 'Function returned the same list object twice'
    assert 'milk (1 gallon)' not in a, 'First list was polluted by second call'
    assert 'apples (2 units)' not in b, 'Second list was polluted by first call'

assert_independent_lists(add_item)
print('test passed')

test passed


## Problem 11 — Production-Ready Grocery API

Write a robust function `add_grocery_item` with validation:

- `name` must be a non-empty string
- `quantity` must be positive
- `unit` must be a non-empty string
- omitted `grocery_list` creates a new list
- supplied `grocery_list` is copied rather than mutated
- returns the updated list

### Solution 11

In [17]:
def add_grocery_item(name, quantity, unit, grocery_list=None):
    if not isinstance(name, str) or not name.strip():
        raise ValueError('name must be a non-empty string')
    if quantity <= 0:
        raise ValueError('quantity must be positive')
    if not isinstance(unit, str) or not unit.strip():
        raise ValueError('unit must be a non-empty string')

    if grocery_list is None:
        grocery_list = []
    else:
        grocery_list = list(grocery_list)

    grocery_list.append(f'{name.strip()} ({quantity} {unit.strip()})')
    return grocery_list

base = ['bread (1 loaf)']
updated = add_grocery_item('milk', 1, 'gallon', base)

print('base:', base)
print('updated:', updated)

try:
    add_grocery_item('', 1, 'unit')
except ValueError as e:
    print(type(e).__name__, e)

base: ['bread (1 loaf)']
updated: ['bread (1 loaf)', 'milk (1 gallon)']
ValueError name must be a non-empty string


## Problem 12 — Final Challenge: Safe Memoized Combinations

Implement `n_choose_k(n, k)` using memoization. Requirements:

- reject negative inputs
- reject `k > n`
- use the identity `C(n, k) = C(n-1, k-1) + C(n-1, k)`
- cache results safely using `lru_cache`
- verify against `math.comb`

### Solution 12

In [18]:
@lru_cache(maxsize=None)
def n_choose_k(n, k):
    if n < 0 or k < 0:
        raise ValueError('n and k must be non-negative')
    if k > n:
        raise ValueError('k cannot be greater than n')
    if k == 0 or k == n:
        return 1
    return n_choose_k(n - 1, k - 1) + n_choose_k(n - 1, k)

for n in range(10):
    for k in range(n + 1):
        assert n_choose_k(n, k) == math.comb(n, k)

print(n_choose_k(30, 15))
print(n_choose_k.cache_info())
print('all tests passed')

155117520
CacheInfo(hits=250, misses=256, maxsize=None, currsize=256)
all tests passed


## Best Practices Summary

1. Do not use mutable objects such as `[]`, `{}`, or `set()` as ordinary default arguments.
2. Use `None` as a default when `None` is not a meaningful input.
3. Use `is None`, not `if not value`, to detect omitted arguments.
4. Use a private sentinel object when `None` is a meaningful input.
5. Copy mutable inputs if your function should not mutate caller-owned objects.
6. Use `deepcopy` for nested structures when full isolation is needed.
7. Intentional mutable defaults can be used for memoization, but they create hidden state.
8. Prefer `functools.lru_cache` for clear, standard memoization.
9. Cache keys must include every input that affects the result.
10. Write tests that check object identity and cross-call pollution.